In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
spark = SparkSession.builder.appName("Week5Coding").getOrCreate()

## Q3. Remove duplicate rows based on user_id and transaction_date

In [4]:
data = [
    (101, "2024-01-01", 5000),
    (101, "2024-01-01", 5000),
    (102, "2024-01-02", 7000)
]

columns = ["user_id", "transaction_date", "amount"]

df_q3 = spark.createDataFrame(data, columns)

df_q3.dropDuplicates(["user_id", "transaction_date"]).show()

+-------+----------------+------+
|user_id|transaction_date|amount|
+-------+----------------+------+
|    101|      2024-01-01|  5000|
|    102|      2024-01-02|  7000|
+-------+----------------+------+



Q4. Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [5]:
from pyspark.sql.functions import avg

sales_data = [
    ("West", "Electronics", 5000),
    ("West", "Furniture", 3000),
    ("West", "Electronics", 7000),
    ("East", "Electronics", 6000)
]

columns = ["region", "product_category", "sale_amount"]

df_sales = spark.createDataFrame(sales_data, columns)

result = (
    df_sales.filter(col("region") == "West")
            .groupBy("product_category")
            .agg(avg("sale_amount").alias("Average_Sale"))
)

result.show()

+----------------+------------+
|product_category|Average_Sale|
+----------------+------------+
|     Electronics|      6000.0|
|       Furniture|      3000.0|
+----------------+------------+



Q5. Provide a code example of filling null values in a status column with the string 'Unknown'.

In [6]:
data = [
    ("Active",),
    (None,),
    ("Inactive",)
]

columns = ["status"]

df_q5 = spark.createDataFrame(data, columns)

df_q5 = df_q5.na.fill({"status": "Unknown"})

df_q5.show()

+--------+
|  status|
+--------+
|  Active|
| Unknown|
|Inactive|
+--------+



Q6. Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [7]:
from pyspark.sql.functions import count

city_data = (
    [("Pune",)] * 120 +
    [("Mumbai",)] * 90 +
    [("Delhi",)] * 150
)

columns = ["city"]

df_q6 = spark.createDataFrame(city_data, columns)

result = (
    df_q6.groupBy("city")
          .agg(count("*").alias("Total_Records"))
          .filter(col("Total_Records") > 100)
)

result.show()

+-----+-------------+
| city|Total_Records|
+-----+-------------+
| Pune|          120|
|Delhi|          150|
+-----+-------------+



Q8. Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [8]:
data = [
    (22, "Premium"),
    (28, "Premium"),
    (35, "Basic"),
    (19, "Premium"),
    (40, "Premium")
]

columns = ["age", "subscription"]

df_q8 = spark.createDataFrame(data, columns)

result = df_q8.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

result.show()

+---+------------+
|age|subscription|
+---+------------+
| 22|     Premium|
| 28|     Premium|
| 19|     Premium|
+---+------------+



Q10. Revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [9]:
from pyspark.sql.types import TimestampType

data = [
    ("2024-01-01 10:30:00",),
    ("2024-02-15 15:45:00",)
]

columns = ["raw_timestamp"]

df_q10 = spark.createDataFrame(data, columns)

df_q10 = (
    df_q10.withColumn(
        "event_time",
        col("raw_timestamp").cast(TimestampType())
    )
    .drop("raw_timestamp")
)

df_q10.show(truncate=False)

+-------------------+
|event_time         |
+-------------------+
|2024-01-01 10:30:00|
|2024-02-15 15:45:00|
+-------------------+



Q12. Identify and remove rows where the email column contains null values OR the username is an empty string.

In [10]:
from pyspark.sql.functions import trim

data = [
    ("rahul@gmail.com", "Rahul"),
    (None, "Amit"),
    ("abc@gmail.com", ""),
    ("xyz@gmail.com", "Sneha")
]

columns = ["email", "username"]

df_q12 = spark.createDataFrame(data, columns)

clean_df = df_q12.filter(
    col("email").isNotNull() &
    (trim(col("username")) != "")
)

clean_df.show()

+---------------+--------+
|          email|username|
+---------------+--------+
|rahul@gmail.com|   Rahul|
|  xyz@gmail.com|   Sneha|
+---------------+--------+



Q13. Use the .agg() function to calculate the minimum, maximum, and average of the price column.

In [11]:
from pyspark.sql.functions import min, max, avg

data = [
    (100,),
    (250,),
    (450,),
    (300,)
]

columns = ["price"]

df_q13 = spark.createDataFrame(data, columns)

df_q13.agg(
    min("price").alias("Minimum_Price"),
    max("price").alias("Maximum_Price"),
    avg("price").alias("Average_Price")
).show()

+-------------+-------------+-------------+
|Minimum_Price|Maximum_Price|Average_Price|
+-------------+-------------+-------------+
|          100|          450|        275.0|
+-------------+-------------+-------------+



Q15. Write a final processing pipeline that:
Filters out duplicates.
Fills null prices with 0.
Groups by store_id to calculate total revenue.

In [12]:
from pyspark.sql.functions import sum

data = [
    (101, 5000),
    (101, None),
    (102, 3000),
    (102, 3000),
    (103, 7000)
]

columns = ["store_id", "price"]

df_q15 = spark.createDataFrame(data, columns)

result = (
    df_q15.dropDuplicates()
          .na.fill({"price": 0})
          .groupBy("store_id")
          .agg(sum("price").alias("Total_Revenue"))
)

result.show()

+--------+-------------+
|store_id|Total_Revenue|
+--------+-------------+
|     103|         7000|
|     101|         5000|
|     102|         3000|
+--------+-------------+

